In [1]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import AgglomerativeClustering




In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "utils").is_dir() and (candidate / "SAE").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
TRAINED_SAE_DIR = REPO_ROOT / "SAE" / "results" / "trained_sae"
ACTIVATIONS_DIR = REPO_ROOT / "SAE" / "results" / "activations"
COFIRE_SUBSAMPLE = 100_000
SEED = 0
pd.set_option("display.max_colwidth", 80)
from utils.sae_utils import DEFAULT_TRAINED_SAE_DIR, sae_dir_name


In [3]:
run_dirs = sorted(d for d in TRAINED_SAE_DIR.iterdir() if d.is_dir() and (d / "config.json").exists())
print(f"{len(run_dirs)} trained SAE runs under {TRAINED_SAE_DIR}")

36 trained SAE runs under c:\Users\chang\OneDrive\Documents\Code\SycoScope\SAE\results\trained_sae


We will find features per layer to steer on. 



In [4]:
LAYER = 18
N_LATENTS = 40
MIN_ACTIVATING = 10

In [12]:
def load_run_latents(layer: int, n_latents: int, k: int = 3, centered: bool = False, seed: int = 0,
                     trained_sae_dir: Path = DEFAULT_TRAINED_SAE_DIR) -> pd.DataFrame:
    """One row per latent for a single trained-SAE run, joining labels.json and metrics.json."""
    run_dir = trained_sae_dir / sae_dir_name(layer, n_latents, k, centered, seed)
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    labels = json.loads((run_dir / "labels.json").read_text(encoding="utf-8"))

    firing_rate = metrics["firing_rate"]
    # steering_p90 comes from compute_steering_coeffs.py and may be absent on a freshly
    # retrained run -- carry NaN rather than failing the whole load.
    p90 = metrics.get("steering_p90")

    rows = []
    for rec in labels:
        lid = rec["latent_id"]
        rows.append({
            "layer": config["layer"],
            "n_latents": config["n_latents"],
            "latent_id": lid,
            "n_activating": rec["n_members_topk"],   # top-k membership
            "cluster_size": rec["cluster_size"],     # argmax-only, kept for contrast
            "firing_rate": firing_rate[lid],
            "steering_p90": p90[lid] if p90 else np.nan,
            "status": rec["status"],
            "title": rec["title"],
            "description": rec["description"],
            "run": sae_dir_name(layer, n_latents, k, centered, seed),
        })
    return pd.DataFrame(rows).sort_values("n_activating", ascending=False).reset_index(drop=True)


latents = load_run_latents(LAYER, N_LATENTS)
print(f"L{LAYER:02d} n={N_LATENTS}: {len(latents)} latents")
# latents[["latent_id", "n_activating", "cluster_size", "firing_rate", "status", "title"]]
latents.head()

L18 n=40: 40 latents


,layer,n_latents,latent_id,n_activating,cluster_size,firing_rate,steering_p90,status,title,description,run
0,18,40,21,201044,60637,0.241262,2.689036,ok,Concluding advice-giving formula for interpersonal conflict resolution,This latent fires on the boilerplate closing-statement pattern that advice-s...,L18_n40_k3_uncentered_s0
1,18,40,33,197352,41979,0.236832,2.428518,ok,Empathetic closing offers of support,"This latent fires on formulaic, supportive closing statements in advice-givi...",L18_n40_k3_uncentered_s0
2,18,40,5,167213,64210,0.200664,3.175025,ok,Empathetic acknowledgment opener,"This latent fires on short, formulaic exclamatory phrases like ""I totally un...",L18_n40_k3_uncentered_s0
3,18,40,30,162540,52059,0.195056,2.365792,ok,Validating self-prioritization/boundary-setting,"This latent fires on affirming, second-person statements that validate the r...",L18_n40_k3_uncentered_s0
4,18,40,12,160791,61324,0.192957,2.146830,ok,Explicit condemnation of someone's reaction as unacceptable,"This latent fires on short, formulaic evaluative statements that pass direct...",L18_n40_k3_uncentered_s0


In [13]:
kept = latents[latents.n_activating >= MIN_ACTIVATING].copy()
dropped = latents[latents.n_activating < MIN_ACTIVATING]


In [14]:
def load_decoder_vector(run_name: str, latent_id: int) -> np.ndarray:
    state_dict = torch.load(TRAINED_SAE_DIR / run_name / "sae.pt", map_location="cpu")
    w_dec = state_dict["W_dec"]  # (d_in, n_latents), columns already unit-norm
    return w_dec[:, latent_id].numpy()

In [15]:
n_sizes = [5,10,15,20,30,40]
latent_dict = {}
for n_size in n_sizes:
    latents = load_run_latents(LAYER, n_size)
    kept = latents[latents.n_activating >= MIN_ACTIVATING].copy()
    latent_dict[str(n_size)] = kept

In [16]:
vectors = load_decoder_vector(sae_dir_name(LAYER, 5), 0)


In [ ]:
TOP_N = 11  # match the table above; bump this to compare more latents

sizes = [5,10]

latents_1 = latent_dict[str(sizes[0])]
latents_2 = latent_dict[str(sizes[1])]

vectors_1 = np.stack([load_decoder_vector(r.run, r.latent_id) for r in latents_1.itertuples()])
vectors_1 = vectors_1 / np.linalg.norm(vectors_1, axis=1, keepdims=True)  # defensive re-normalize

vectors_2 = np.stack([load_decoder_vector(r.run, r.latent_id) for r in latents_2.itertuples()])
vectors_2 = vectors_2 / np.linalg.norm(vectors_2, axis=1, keepdims=True)  # defensive re-normalize
cos_sim = vectors @ vectors.T

array([[ 0.9999999 , -0.00138982, -0.01358029,  0.19231968,  0.04916972,
         0.1105265 ,  0.13560611,  0.12975626,  0.17677988,  0.04370539,
         0.0730066 ],
       [-0.00138982,  0.9999999 ,  0.09649495,  0.05116612,  0.10836834,
         0.03727837,  0.24215654,  0.14144123,  0.16973571,  0.21422407,
         0.23633045],
       [-0.01358029,  0.09649495,  1.        ,  0.10874177,  0.05886434,
        -0.16676413,  0.09048054,  0.20963012,  0.11227015,  0.08305933,
         0.29351887],
       [ 0.19231968,  0.05116612,  0.10874177,  0.99999994,  0.26915175,
         0.06457306,  0.21264035,  0.45893976,  0.51257575,  0.28481412,
         0.20641984],
       [ 0.04916972,  0.10836834,  0.05886434,  0.26915175,  0.9999998 ,
        -0.09850461,  0.31693238,  0.09219864,  0.19056019,  0.15643266,
         0.1037966 ],
       [ 0.1105265 ,  0.03727837, -0.16676413,  0.06457306, -0.09850461,
         0.99999994,  0.06482351,  0.03827754,  0.1266114 ,  0.17894286,
         0.081